# **Getting the data by upload it manually since it's small**
www.kaggle.com/datasets/sinjoysaha/sales-analysis-dataset


# **Prepare The Data** **ℾ**

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import joblib

# Getting the data
df = pd.read_csv("/kaggle/input/datasets/ouicifarouk/data-electronics/ready_electonic_sales_data.csv")

# transform the date column into pandas date-type
df['Date'] = pd.to_datetime(df['Date'])
df['Week'] = df['Date'].dt.isocalendar().week

# combining the rows based on some features
df_new = df.groupby(["Product","Year","Week","Day"]).agg(
    Quantity_Ordered=("Quantity Ordered", "sum"), # here we sum the quantity oredered
    Price_Each = ("Price Each","mean"),
    ).reset_index() # to not return the columns index

print(df_new)

# Saving the dataframe 
df_new.to_csv("electronic-store-sales_cleaned.csv",index=False)

# we split the data based on conditions to overcome the issue of future leakage
train = df_new[(df_new["Week"] < 38) & (df_new["Year"] < 2020)] 
test = df_new[(df_new["Week"] >= 38) & (df_new["Year"] == 2019) | (df_new["Year"] == 2020)]

# Spliting the data to train and test
X_train = train.drop("Quantity_Ordered",axis=1).copy()
y_train = train["Quantity_Ordered"].copy()

X_test = test.drop("Quantity_Ordered",axis=1).copy()
y_test = test["Quantity_Ordered"].copy()

# Making the months cyclical
for df_X in [X_train, X_test]:
    df_X["Week_sin"] = np.sin(2.0 * np.pi * df_X["Week"] / 52.0)
    df_X["Week_cos"] = np.cos(2.0 * np.pi * df_X["Week"] / 52.0)

print("The Data Has been splited")

print(X_train.head())
print(X_train.shape)
print(X_train["Product"].value_counts())

# **Applying the techniques** ∮

In [ ]:
import time
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import FunctionTransformer,StandardScaler,OneHotEncoder
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


X_train["Price_Each"] = X_train["Price_Each"].round()
X_test["Price_Each"] = X_test["Price_Each"].round()

one_hot_col = ["Product"]
log_col = ["Price_Each"]


log_function = FunctionTransformer(np.log10)

one_hot_pipeline = Pipeline([
    ("onehot",OneHotEncoder(sparse_output=False,handle_unknown="ignore")),
])

log_pipeline = Pipeline([
    ("log_transformer", log_function), 
     ("scaler", StandardScaler())
])


preprocessor = ColumnTransformer(
    transformers=[
        ("one_hot", one_hot_pipeline, one_hot_col),  
        ("log", log_pipeline, log_col),
    ]
)


linear_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "regressor",
            LinearRegression(),
        ), 
    ]
)

t = time.time()
model = linear_pipeline.fit(X_train,y_train)
t_a = time.time()
print(f'the model was trained on :  {(t_a - t):.4f} s')

y_pred = model.predict(X_test)

scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
print(f"Mean CV R²: {scores.mean():.4f} (+/- {scores.std():.4f})")



mse_a = mean_squared_error(y_test, y_pred)
rmse_a= np.sqrt(mse_a)
mae_a = mean_absolute_error(y_test, y_pred)
r2_a = r2_score(y_test, y_pred)

y_t_pred = model.predict(X_train)

r2_t = r2_score(y_train, y_t_pred)
mae_t = mean_absolute_error(y_train, y_t_pred)
rmse_t = np.sqrt(mean_squared_error(y_train, y_t_pred))

print("\n=== Train vs Test Performance ===")
print(f"Train R² : {r2_t:.4f}  |  Test R² : {r2_a:.4f}")
print(f"Train MAE: {mae_t:.4f}  |  Test MAE: {mae_a:.4f}")
print(f"Train RMSE: {rmse_t:.4f} |  Test RMSE: {rmse_a:.4f}")


# Saving the model

joblib.dump(model,"DemandMind.pkl")

In [ ]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV,HalvingRandomSearchCV
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor




model_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "regressor",
            XGBRegressor(random_state=42, n_jobs=-1),
        ), 
    ]
)


parms_distrubtion = {
    "regressor__max_depth": [5, 7, 9, 11],
    "regressor__n_estimators": [100, 150, 200, 250],
    "regressor__learning_rate": [0.01,0.05,0.1,0.2],
    "regressor__subsample": [0.4,0.6,0.8,1.0],
    "regressor__colsample_bytree": [0.4,0.6,0.8,1.0],
}

print("Start The Random Search....")
halving_random = HalvingRandomSearchCV(
    estimator=model_pipeline,
    param_distributions=parms_distrubtion,
    factor=3,
    resource="n_samples",
    max_resources="auto",
    cv=3,
    random_state=42,
    n_jobs=1,  
)

t_s = time.time()
halving_random.fit(X_train, y_train)
t_e = time.time()
print("Finished!")
print(
    f"The best parameters is : {halving_random.best_params_} founded in :"
    f"  {(t_e - t_s):.4f} s"
)

best_depth = halving_random.best_params_["regressor__max_depth"]
best_est = halving_random.best_params_["regressor__n_estimators"]

param_grid_fine = {
    "regressor__n_estimators": [
        max(30, best_est - 30),
        best_est,
        best_est + 30,
    ],
    "regressor__max_depth": [best_depth - 1, best_depth, best_depth + 1],
    "regressor__learning_rate": [
        halving_random.best_params_["regressor__learning_rate"]
    ],
    "regressor__subsample": [
        halving_random.best_params_["regressor__subsample"]
    ],
    "regressor__colsample_bytree": [
        halving_random.best_params_["regressor__colsample_bytree"]
    ],
}

print("Starting the GridSearch....")
halving_grid = HalvingGridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid_fine,
    factor=3,
    cv=3,
    n_jobs=1,
)

time_st = time.time()
halving_grid.fit(X_train, y_train)
time_en = time.time()

best_model = halving_grid.best_estimator_
print(
    f"The model was fine-tuned successfully in : {(time_en - time_st):.4f} s"
)

scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2')
print(f"Mean CV R²: {scores.mean():.4f} (+/- {scores.std():.4f})")


model_pred = best_model.predict(X_test)

mse = mean_squared_error(y_test, model_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, model_pred)
r2 = r2_score(y_test, model_pred)

y_train_pred = best_model.predict(X_train)

r2_train = r2_score(y_train, y_train_pred)
mae_train = mean_absolute_error(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))

print("\n=== Train vs Test Performance ===")
print(f"Train R² : {r2_train:.4f}  |  Test R² : {r2:.4f}")
print(f"Train MAE: {mae_train:.4f}  |  Test MAE: {mae:.4f}")
print(f"Train RMSE: {rmse_train:.4f} |  Test RMSE: {rmse:.4f}")